In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import joblib

print("REHBERE UYGUN ADIM 1: VERİ HAZIRLAMA VE SPLIT ")

# 1. Target ve Özellik Ayrımı
target_col = "isFraud"
y = train_df[target_col].astype(int)
# TransactionID'yi kılavuz uyarısı doğrultusunda çıkarıyoruz
X = train_df.drop(columns=[target_col, "TransactionID"])

# 2. Üçlü Split (%70 Train, %15 Val, %15 Test) - Stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train size: {X_train.shape} | Val size: {X_val.shape} | Test size: {X_test.shape}")

# 3. Sayısal ve Kategorik Kolonların Otomatik Ayrımı
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

# 4. Preprocessing Pipeline Yapısının Kurulması
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # Sayısal -> Median ile doldur
    ("scaler", StandardScaler()) # Sayısal -> Ölçeklendir
])

categorical_pipeline = Pipeline([
    # Kategorik -> En sık değerle doldur ve One-Hot encoding uygula
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 5. VERİ SIZINTISINI ÖNLEME: Fit işlemi SADECE Train setinde yapılır!
print("Preprocessing sadece Train verisine fit ediliyor...")
X_train_processed = preprocessor.fit_transform(X_train)
# Validation ve Test setleri sadece transform edilir!
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


print("\nREHBERE UYGUN ADIM 2: HAFİF MODEL (KÜÇÜK RANDOM FOREST)")

# 6. Hafif Model Eğitimi (Slayt ve rehbere uygun küçük RF)
light_model = RandomForestClassifier(
    n_estimators=50, 
    max_depth=5, 
    class_weight="balanced", 
    random_state=42,
    n_jobs=-1
)
light_model.fit(X_train_processed, y_train)
print("Hafif Model başarıyla eğitildi.")

# 7. Validation Tahminlerinin Üretilmesi
val_probs = light_model.predict_proba(X_val_processed)[:, 1]

# 8. Threshold Tuning (Eşik Değer Analizi)
print("\nThreshold Analiz Raporu:")
rows = []
for threshold in np.arange(0.05, 0.55, 0.05):
    pred = (val_probs >= threshold).astype(int)
    rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "routed_rate": pred.mean() # Ağır modele gidecek oran
    })
threshold_results = pd.DataFrame(rows)
print(threshold_results)

# 9. Rehbere uygun olarak en az %90 Recall veren dengeli bir threshold seçiyoruz
selected_threshold = 0.15 
print(f"\nSeçilen Karar Eşiği (Threshold): {selected_threshold}")

# 10. Rehber uyarınca Pipeline, Kolonlar ve Threshold'u Tek Paket Olarak Kaydetme
joblib.dump(
    {
        "preprocessor": preprocessor,
        "model": light_model,
        "threshold": selected_threshold,
        "feature_columns": X_train.columns.tolist()
    },
    "best_light_model_bundle.joblib"
)
print("Hafif model ve tüm bileşenleri başarıyla paketlenerek kaydedildi! (best_light_model_bundle.joblib)")

REHBERE UYGUN ADIM 1: VERİ HAZIRLAMA VE SPLIT 
Train size: (413378, 432) | Val size: (88581, 432) | Test size: (88581, 432)
Preprocessing sadece Train verisine fit ediliyor...


/var/folders/k5/m0ckcyj91zg3bjr1k8mvd02m0000gn/T/ipykernel_27924/2299988956.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()



REHBERE UYGUN ADIM 2: HAFİF MODEL (KÜÇÜK RANDOM FOREST)
Hafif Model başarıyla eğitildi.

Threshold Analiz Raporu:
   threshold  precision    recall        f1  routed_rate
0       0.05   0.034996  1.000000  0.067626     1.000000
1       0.10   0.035002  1.000000  0.067636     0.999842
2       0.15   0.037097  0.997419  0.071534     0.940924
3       0.20   0.045673  0.964516  0.087215     0.739052
4       0.25   0.047980  0.954839  0.091368     0.696459
5       0.30   0.049442  0.950645  0.093996     0.672887
6       0.35   0.050607  0.941290  0.096050     0.650930
7       0.40   0.084276  0.831613  0.153042     0.345334
8       0.45   0.103686  0.762258  0.182542     0.257279
9       0.50   0.117742  0.707742  0.201896     0.210361

Seçilen Karar Eşiği (Threshold): 0.15
Hafif model ve tüm bileşenleri başarıyla paketlenerek kaydedildi! (best_light_model_bundle.joblib)


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import joblib

print(" REHBERE UYGUN ADIM 3: AĞIR MODEL (GÜÇLÜ RANDOM FOREST) EĞİTİMİ")

# 1. Sınıf Dengesizliği Oranını (scale_pos_weight benzeri) Hesaplama
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
print(f"Normal İşlem Sayısı: {negative_count}")
print(f"Fraud İşlem Sayısı: {positive_count}")

# 2. Ağır Modelin (Güçlü Random Forest) Tanımlanması
# Hafif modele kıyasla çok daha fazla ağaç (150) ve derinlik (12) veriyoruz ki detayları kaçırmasın![cite: 1]
print("\nAğır Model (Random Forest) eğitiliyor, lütfen bekleyin...")
heavy_model = RandomForestClassifier(
    n_estimators=150, 
    max_depth=12, 
    class_weight="balanced", # Sınıf dengesizliğini çözmek için[cite: 1]
    random_state=42,
    n_jobs=-1 # Mac'inin tüm gücünü kullanarak çok hızlı eğitir
)

# Eğitiyoruz
heavy_model.fit(X_train_processed, y_train)
print("Ağır Model eğitimi tamamlandı!")

# 3. Validation Seti Üzerinde Tahmin ve Performans Ölçümü[cite: 1]
heavy_probs = heavy_model.predict_proba(X_val_processed)[:, 1]
heavy_pr_auc = average_precision_score(y_val, heavy_probs)

print("\n================ AĞIR MODEL BAŞARISI ================")
print(f"Ağır Model (Random Forest) Validation PR-AUC Skoru: {heavy_pr_auc:.4f}")
print("=====================================================")

# 4. Rehber uyarınca Ağır Modeli de Paket Olarak Kaydetme[cite: 1]
joblib.dump(
    {
        "model": heavy_model,
        "preprocessor": preprocessor,
        "feature_columns": X_train.columns.tolist()
    },
    "heavy_model_bundle.joblib"
)
print("\nAğır model ve bileşenleri başarıyla kaydedildi! (heavy_model_bundle.joblib)")

 REHBERE UYGUN ADIM 3: AĞIR MODEL (GÜÇLÜ RANDOM FOREST) EĞİTİMİ
Normal İşlem Sayısı: 398914
Fraud İşlem Sayısı: 14464

Ağır Model (Random Forest) eğitiliyor, lütfen bekleyin...
Ağır Model eğitimi tamamlandı!

================ AĞIR MODEL BAŞARISI ================
Ağır Model (Random Forest) Validation PR-AUC Skoru: 0.5332

Ağır model ve bileşenleri başarıyla kaydedildi! (heavy_model_bundle.joblib)


In [14]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, average_precision_score, recall_score

print("REHBERE UYGUN FİNAL ADIMI: ECOSHIELD AI CASCADE SIMÜLASYONU")

# 1. Kaydettiğimiz bundle paketlerini diskten geri yüklüyoruz
print("Hafif ve Ağır model paketleri hafızaya alınıyor...")
light_bundle = joblib.load("best_light_model_bundle.joblib")
heavy_bundle = joblib.load("heavy_model_bundle.joblib")

# Paketlerin içindeki eğitilmiş modelleri ve preprocessing nesnelerini çıkarıyoruz
light_model = light_bundle["model"]
preprocessor = light_bundle["preprocessor"] # Ortak preprocessing nesnemiz
selected_threshold = light_bundle["threshold"]

heavy_model = heavy_bundle["model"]

# 2. Aşama 1: Hafif modelimiz TÜM test verisini saniyeler içinde tarıyor
print("\n1. Aşama: Hafif model (Random Forest) test verisini tarıyor...")
# Önceden işlediğimiz test verisini doğrudan kullanıyoruz
light_test_probs = light_model.predict_proba(X_test_processed)[:, 1]

# 3. Aşama 2: Hafif modelin seçtiği şüpheli (riskli) işlemleri belirliyoruz
# Hafif modelde belirlediğimiz eşik değerini (0.15) baz alıyoruz
riskli_islem_indeksleri = np.where(light_test_probs >= selected_threshold)[0]

# 4. Aşama 3: Sadece bu şüpheli işlemleri detaylı inceleme için Ağır Modele gönderiyoruz
print(f"2. Aşama: Belirlenen şüpheli işlemler Ağır Modele (Derin Random Forest) sevk ediliyor...")
X_test_heavy = X_test_processed[riskli_islem_indeksleri]

# Eğer ağır modele yönlendirilen işlem varsa tahmin üretiyoruz
if len(X_test_heavy) > 0:
    heavy_test_probs = heavy_model.predict_proba(X_test_heavy)[:, 1]
    
    # Tüm test seti için nihai tahmin dizisini oluşturuyoruz
    # Hafif modelin güvendiği (risk skoru < 0.15) işlemlere direkt 0 (Normal) diyoruz
    final_predictions = np.zeros(len(X_test_processed))
    
    # Ağır modelin karar eşiğini de standart 0.50 olarak alıp şüphelileri etiketliyoruz
    final_predictions[riskli_islem_indeksleri] = (heavy_test_probs >= 0.50).astype(int)
else:
    final_predictions = np.zeros(len(X_test_processed))

# 5. Tasarruf ve Performans Hesaplamaları
toplam_islem = len(X_test_processed)
agir_modele_giden = len(X_test_heavy)
compute_saving = 1 - (agir_modele_giden / toplam_islem) # Rehberdeki tasarruf formülü

# Genel sistem başarısını ölçüyoruz
final_recall = recall_score(y_test, final_predictions)

print("\n================ ECOSHIELD AI FİNAL RAPORU ================")
print(f"Sisteme Gelen Toplam İşlem Sayısı (Test Seti): {toplam_islem}")
print(f"Hafif Modeli Geçemeyip Ağır Modele Sevk Edilen: {agir_modele_giden}")
print(f"🔥 COMPUTE-SAVING (Hesaplama Gücü Tasarrufu): %{compute_saving * 100:.2f}")
print(f"🎯 KORUNAN SİSTEM RECALL DEĞERİ: %{final_recall * 100:.2f}")
print("===========================================================")

REHBERE UYGUN FİNAL ADIMI: ECOSHIELD AI CASCADE SIMÜLASYONU
Hafif ve Ağır model paketleri hafızaya alınıyor...

1. Aşama: Hafif model (Random Forest) test verisini tarıyor...
2. Aşama: Belirlenen şüpheli işlemler Ağır Modele (Derin Random Forest) sevk ediliyor...

================ ECOSHIELD AI FİNAL RAPORU ================
Sisteme Gelen Toplam İşlem Sayısı (Test Seti): 88581
Hafif Modeli Geçemeyip Ağır Modele Sevk Edilen: 83230
🔥 COMPUTE-SAVING (Hesaplama Gücü Tasarrufu): %6.04
🎯 KORUNAN SİSTEM RECALL DEĞERİ: %72.54


In [15]:
# Karar eşiğini (Threshold) 0.35'e yükseltiyoruz!
yeni_threshold = 0.35

# 1. Hafif model yeniden tahmin üretiyor
light_test_probs = light_model.predict_proba(X_test_processed)[:, 1]

# 2. Artık sadece %35 ve üzeri riskli olanları ağır modele sevk ediyoruz
riskli_islem_yeni = np.where(light_test_probs >= yeni_threshold)[0]
X_test_heavy_yeni = X_test_processed[riskli_islem_yeni]

# 3. Ağır model sadece bu azalan şüphelileri inceliyor
heavy_test_probs_yeni = heavy_model.predict_proba(X_test_heavy_yeni)[:, 1]

final_predictions_yeni = np.zeros(len(X_test_processed))
final_predictions_yeni[riskli_islem_yeni] = (heavy_test_probs_yeni >= 0.50).astype(int)

# 4. Yeni Tasarruf Oranları
toplam_islem = len(X_test_processed)
agir_modele_giden_yeni = len(X_test_heavy_yeni)
compute_saving_yeni = 1 - (agir_modele_giden_yeni / toplam_islem)

print("\n================ ECOSHIELD AI YENİ STRATEJİ RAPORU ================")
print(f"Sisteme Gelen Toplam İşlem: {toplam_islem}")
print(f"Ağır Modele (Dedektife) Sevk Edilen İşlem: {agir_modele_giden_yeni}")
print(f"🔥 YENİ COMPUTE-SAVING (Hesaplama Tasarrufu): %{compute_saving_yeni * 100:.2f}")
print("===================================================================")


================ ECOSHIELD AI YENİ STRATEJİ RAPORU ================
Sisteme Gelen Toplam İşlem: 88581
Ağır Modele (Dedektife) Sevk Edilen İşlem: 57604
🔥 YENİ COMPUTE-SAVING (Hesaplama Tasarrufu): %34.97
